# XL-Share: Comprehensive Evaluation for Manuscript

This notebook implements a rigorous evaluation of the **XL-Share** system, suitable for a research manuscript.

## Experimental Design
We compare three approaches:
1.  **Ideal (Upper Bound)**: All weights pinned in GPU memory (simulated if OOM).
2.  **Naive Offloading (Baseline)**: Synchronous copy of weights from CPU to GPU on demand.
3.  **XL-Share (Ours)**: Intelligent Asynchronous Prefetching + LRU Caching.

## Independent Variables
- **Cache Size**: Impact of GPU memory constraints (128MB, 256MB, 512MB).
- **Batch Size**: Impact of computational intensity.
- **Model Size**: Scaling behavior.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import threading
import queue
import matplotlib.pyplot as plt
from collections import OrderedDict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {DEVICE}")

## 1. Core System Components (Memory & Cache)

In [ ]:
class CXLMemoryManager:
    def __init__(self):
        self.pool: Dict[str, torch.Tensor] = {}
        self.total_bytes = 0

    def store_weight(self, name: str, tensor: torch.Tensor):
        self.pool[name] = tensor.cpu().pin_memory()
        self.total_bytes += tensor.numel() * tensor.element_size()

    def get_weight(self, name: str) -> torch.Tensor:
        return self.pool[name]

class LocalCache:
    def __init__(self, capacity_mb: int = 1024):
        self.capacity_bytes = capacity_mb * 1024 * 1024
        self.current_bytes = 0
        self.cache: OrderedDict[str, torch.Tensor] = OrderedDict()
        self.lock = threading.RLock()
        self.evictions = 0
        self.hits = 0
        self.misses = 0

    def get(self, name: str) -> Optional[torch.Tensor]:
        with self.lock:
            if name in self.cache:
                tensor = self.cache.pop(name)
                self.cache[name] = tensor
                self.hits += 1
                return tensor
            self.misses += 1
            return None

    def put(self, name: str, tensor: torch.Tensor):
        with self.lock:
            size = tensor.numel() * tensor.element_size()
            if name in self.cache:
                old_tensor = self.cache.pop(name)
                self.current_bytes -= (old_tensor.numel() * old_tensor.element_size())
            
            while self.current_bytes + size > self.capacity_bytes and len(self.cache) > 0:
                self.cache.popitem(last=False)
                self.current_bytes -= (tensor.numel() * tensor.element_size()) # Approximation for simplicity
                self.evictions += 1
            
            self.cache[name] = tensor
            self.current_bytes += size

    def reset_stats(self):
        self.hits = 0
        self.misses = 0
        self.evictions = 0

## 2. Engines: Naive vs. XL-Share

In [ ]:
def functional_linear(input, weight, bias=None):
    return F.linear(input, weight, bias)

def functional_layer_norm(input, weight, bias, normalized_shape):
    return F.layer_norm(input, normalized_shape, weight, bias)

class BaseEngine:
    def __init__(self, hidden_size, num_layers, vocab_size, cache_size_mb):
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.vocab_size = vocab_size
        self.mem_manager = CXLMemoryManager()
        self.local_cache = LocalCache(capacity_mb=cache_size_mb)
        self._init_weights()
        self.layer_names = self._get_execution_order()

    def _init_weights(self):
        # Initialize weights (same as before)
        self.mem_manager.store_weight("emb.weight", torch.randn(self.vocab_size, self.hidden_size))
        for i in range(self.num_layers):
            self.mem_manager.store_weight(f"l{i}.ln1.weight", torch.ones(self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ln1.bias", torch.zeros(self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.attn.q.weight", torch.randn(self.hidden_size, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.attn.k.weight", torch.randn(self.hidden_size, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.attn.v.weight", torch.randn(self.hidden_size, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.attn.o.weight", torch.randn(self.hidden_size, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ln2.weight", torch.ones(self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ln2.bias", torch.zeros(self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ff1.weight", torch.randn(self.hidden_size * 4, self.hidden_size))
            self.mem_manager.store_weight(f"l{i}.ff2.weight", torch.randn(self.hidden_size, self.hidden_size * 4))
        self.mem_manager.store_weight("head.weight", torch.randn(self.vocab_size, self.hidden_size))

    def _get_execution_order(self):
        order = ["emb.weight"]
        for i in range(self.num_layers):
            order.extend([
                f"l{i}.ln1.weight", f"l{i}.ln1.bias",
                f"l{i}.attn.q.weight", f"l{i}.attn.k.weight", f"l{i}.attn.v.weight", f"l{i}.attn.o.weight",
                f"l{i}.ln2.weight", f"l{i}.ln2.bias",
                f"l{i}.ff1.weight", f"l{i}.ff2.weight"
            ])
        order.append("head.weight")
        return order

    def get_weight(self, name: str) -> torch.Tensor:
        raise NotImplementedError

    def forward(self, x_input):
        # Generic forward pass using get_weight()
        w_emb = self.get_weight("emb.weight")
        x = F.embedding(x_input, w_emb)
        
        for i in range(self.num_layers):
            w_ln1 = self.get_weight(f"l{i}.ln1.weight")
            b_ln1 = self.get_weight(f"l{i}.ln1.bias")
            residual = x
            x = functional_layer_norm(x, w_ln1, b_ln1, (self.hidden_size,))
            
            w_q = self.get_weight(f"l{i}.attn.q.weight")
            w_k = self.get_weight(f"l{i}.attn.k.weight")
            w_v = self.get_weight(f"l{i}.attn.v.weight")
            w_o = self.get_weight(f"l{i}.attn.o.weight")
            
            q = functional_linear(x, w_q)
            k = functional_linear(x, w_k)
            v = functional_linear(x, w_v)
            # Simplified attention
            x = functional_linear(q, w_o) + residual
            
            w_ln2 = self.get_weight(f"l{i}.ln2.weight")
            b_ln2 = self.get_weight(f"l{i}.ln2.bias")
            residual = x
            x = functional_layer_norm(x, w_ln2, b_ln2, (self.hidden_size,))
            
            w_ff1 = self.get_weight(f"l{i}.ff1.weight")
            w_ff2 = self.get_weight(f"l{i}.ff2.weight")
            x = functional_linear(x, w_ff1)
            x = F.relu(x)
            x = functional_linear(x, w_ff2) + residual
            
        w_head = self.get_weight("head.weight")
        return functional_linear(x, w_head)


class NaiveOffloadingEngine(BaseEngine):
    """Baseline: Synchronous copy on demand"""
    def get_weight(self, name: str) -> torch.Tensor:
        # Check cache
        tensor = self.local_cache.get(name)
        if tensor is not None:
            return tensor
        
        # Synchronous copy
        cpu_tensor = self.mem_manager.get_weight(name)
        gpu_tensor = cpu_tensor.to(DEVICE)
        self.local_cache.put(name, gpu_tensor)
        return gpu_tensor


class XLShareEngine(BaseEngine):
    """Ours: Async Prefetching"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.stream = torch.cuda.Stream() if torch.cuda.is_available() else None
        self.active_transfers = {}
        self.curr_idx = 0

    def prefetch(self, lookahead=5):
        if not self.stream: return
        
        end_idx = min(len(self.layer_names), self.curr_idx + lookahead)
        with torch.cuda.stream(self.stream):
            for name in self.layer_names[self.curr_idx:end_idx]:
                if self.local_cache.get(name) is not None: continue
                
                cpu_tensor = self.mem_manager.get_weight(name)
                gpu_tensor = cpu_tensor.to(DEVICE, non_blocking=True)
                self.local_cache.put(name, gpu_tensor)
                event = torch.cuda.Event()
                event.record(self.stream)
                self.active_transfers[name] = event

    def get_weight(self, name: str) -> torch.Tensor:
        # Trigger prefetch
        self.prefetch()
        self.curr_idx += 1
        
        tensor = self.local_cache.get(name)
        if tensor is not None:
            if name in self.active_transfers:
                self.active_transfers[name].wait()
                del self.active_transfers[name]
            return tensor
            
        # Fallback
        cpu_tensor = self.mem_manager.get_weight(name)
        gpu_tensor = cpu_tensor.to(DEVICE)
        self.local_cache.put(name, gpu_tensor)
        return gpu_tensor
    
    def forward(self, x_input):
        self.curr_idx = 0
        return super().forward(x_input)

## 3. Experiment Runner

In [ ]:
def run_experiment(engine_cls, cache_size_mb, batch_size=4, num_iters=5):
    engine = engine_cls(
        hidden_size=1024, 
        num_layers=12, 
        vocab_size=50000, 
        cache_size_mb=cache_size_mb
    )
    
    x_input = torch.randint(0, 50000, (batch_size, 128)).to(DEVICE)
    
    # Warmup
    engine.forward(x_input)
    torch.cuda.synchronize()
    engine.local_cache.reset_stats()
    
    # Measure
    start = time.time()
    for _ in range(num_iters):
        engine.forward(x_input)
        torch.cuda.synchronize()
    total_time = time.time() - start
    
    avg_latency = (total_time / num_iters) * 1000
    throughput = (num_iters * batch_size * 128) / total_time
    hit_rate = engine.local_cache.hits / (engine.local_cache.hits + engine.local_cache.misses + 1e-6)
    
    return avg_latency, throughput, hit_rate

results = {
    'naive': {'latencies': [], 'throughputs': [], 'hit_rates': []},
    'xlshare': {'latencies': [], 'throughputs': [], 'hit_rates': []}
}

cache_sizes = [128, 256, 512, 1024]

print("Running Experiments...")
for size in cache_sizes:
    print(f"\nTesting Cache Size: {size}MB")
    
    # Naive
    lat, tpt, hr = run_experiment(NaiveOffloadingEngine, size)
    results['naive']['latencies'].append(lat)
    results['naive']['throughputs'].append(tpt)
    print(f"  Naive:   {lat:.1f}ms, {tpt:.0f} tok/s, Hit Rate: {hr:.2f}")
    
    # XL-Share
    lat, tpt, hr = run_experiment(XLShareEngine, size)
    results['xlshare']['latencies'].append(lat)
    results['xlshare']['throughputs'].append(tpt)
    print(f"  XL-Share: {lat:.1f}ms, {tpt:.0f} tok/s, Hit Rate: {hr:.2f}")

## 4. Visualization

In [ ]:
plt.figure(figsize=(12, 5))

# Latency Plot
plt.subplot(1, 2, 1)
plt.plot(cache_sizes, results['naive']['latencies'], 'o--', label='Naive Offloading', color='gray')
plt.plot(cache_sizes, results['xlshare']['latencies'], 'o-', label='XL-Share (Ours)', color='blue', linewidth=2)
plt.xlabel('GPU Cache Size (MB)')
plt.ylabel('Inference Latency (ms)')
plt.title('Latency vs. Cache Size')
plt.legend()
plt.grid(True, alpha=0.3)

# Throughput Plot
plt.subplot(1, 2, 2)
plt.plot(cache_sizes, results['naive']['throughputs'], 'o--', label='Naive Offloading', color='gray')
plt.plot(cache_sizes, results['xlshare']['throughputs'], 'o-', label='XL-Share (Ours)', color='blue', linewidth=2)
plt.xlabel('GPU Cache Size (MB)')
plt.ylabel('Throughput (tokens/sec)')
plt.title('Throughput vs. Cache Size')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()